In [75]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lower, col, explode, split, row_number
from pyspark.sql.window import Window
import re
import html

spark = SparkSession.builder.appName('Lab2').master('local').getOrCreate()

In [76]:
languages = spark.read.csv('/content/programming-languages.csv', header=True)
languages = languages.select(lower(languages.name).alias('name'))

posts = spark.sparkContext.textFile('posts_sample.xml')

def extract_tags(line):
  return re.findall(r'<([^>]+)>', html.unescape(line))

def extract_year(line):
  return int(line.split('-')[0])

def extract_data(line):
  if not line.strip().startswith('<row'):
    return None

  year_match = re.search(r'CreationDate="([^"]+)"', line)
  if not year_match:
      return None
  year = extract_year(year_match.group(1))

  tags_match = re.search(r'Tags="([^"]+)"', line)
  if not tags_match:
    return None
  tags = extract_tags(tags_match.group(1))

  return (year, tags)

posts_data = posts \
  .map(extract_data) \
  .filter(lambda x: x is not None) \
  .toDF(['year', 'tags'])


processed_data = posts_data \
  .filter((col("year") >= "2010") & (col("year") <= "2020")) \
  .withColumn("tag", explode(col("tags"))) \
  .withColumn("tag", lower(col("tag"))) \
  .join(languages, col("tag") == col("name")) \
  .groupBy("year", "tag") \
  .count()

result = processed_data \
  .withColumn("rank", row_number().over(Window.partitionBy("year").orderBy(col("count").desc()))) \
  .filter(col("rank") <= 10) \
  .drop("rank")

result.coalesce(1).write.mode("overwrite").parquet("result.parquet")
print(result.collect())

[Row(year=2010, tag='java', count=52), Row(year=2010, tag='php', count=46), Row(year=2010, tag='javascript', count=44), Row(year=2010, tag='python', count=26), Row(year=2010, tag='objective-c', count=23), Row(year=2010, tag='c', count=20), Row(year=2010, tag='ruby', count=12), Row(year=2010, tag='delphi', count=8), Row(year=2010, tag='bash', count=3), Row(year=2010, tag='r', count=3), Row(year=2011, tag='php', count=102), Row(year=2011, tag='java', count=93), Row(year=2011, tag='javascript', count=83), Row(year=2011, tag='python', count=37), Row(year=2011, tag='objective-c', count=34), Row(year=2011, tag='c', count=24), Row(year=2011, tag='ruby', count=20), Row(year=2011, tag='perl', count=9), Row(year=2011, tag='delphi', count=8), Row(year=2011, tag='bash', count=7), Row(year=2012, tag='php', count=154), Row(year=2012, tag='javascript', count=132), Row(year=2012, tag='java', count=124), Row(year=2012, tag='python', count=69), Row(year=2012, tag='objective-c', count=45), Row(year=2012,